<a href="https://colab.research.google.com/github/adigenova/uohpmd/blob/main/code/Haddop_nextflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hadoop

## Calcular el promedio

In [1]:
%%writefile mapper1.py
#!/usr/bin/env python
import sys

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    number = float(line.strip())
    # Emite el número como clave y valor
    print("Total", number)
    print("Count", 1)

Writing mapper1.py


In [ ]:
!chmod 777 /content/mapper1.py /content/reducer1.py

chmod: cannot access '/content/reducer1.py': No such file or directory


In [2]:
%%writefile reducer1.py
#!/usr/bin/env python
import sys

current_key = None
total = 0
count = 0

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    key, value = line.strip().split(" ")
    value = float(value)

    if key == "Total":
        total += value
    elif key == "Count":
        count += int(value)

# Calcula el promedio
average = total / count
print("Average", average)


Writing reducer1.py


In [9]:
!seq 1 1000 | python mapper1.py | sort -k1,1 | python reducer1.py

Average 500.5


In [7]:
!seq 1  10 | python mapper1.py | sort -k1,1

Count 1
Count 1
Count 1
Count 1
Count 1
Count 1
Count 1
Count 1
Count 1
Count 1
Total 1.0
Total 10.0
Total 2.0
Total 3.0
Total 4.0
Total 5.0
Total 6.0
Total 7.0
Total 8.0
Total 9.0


## ventas por producto

In [10]:
%%writefile data.txt
Fecha,Producto,Cantidad,PrecioUnitario
2023-10-23,ProductoA,5,10.0
2023-10-23,ProductoB,3,15.0
2023-10-24,ProductoA,2,10.0
2023-10-24,ProductoC,1,20.0
2023-10-25,ProductoB,4,15.0
2023-10-25,ProductoC,2,20.0
2023-10-26,ProductoA,3,10.0
2023-10-26,ProductoB,2,15.0
2023-10-27,ProductoC,3,20.0
2023-10-27,ProductoD,1,25.0


Writing data.txt


In [11]:
%%writefile mapper2.py

#!/usr/bin/env python
import sys

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    # Divide la línea en campos
    fields = line.strip().split(",")
    if len(fields) == 4:
        date, product, quantity, price = fields
        # Emite el producto como clave y la cantidad como valor
        print(f"{product}\t{quantity}")


Writing mapper2.py


In [12]:
%%writefile reducer2.py

#!/usr/bin/env python
import sys

current_product = None
total_quantity = 0

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    product, quantity = line.strip().split("\t")
    quantity = int(quantity)

    if product == current_product:
        total_quantity += quantity
    else:
        if current_product:
            print(f"{current_product}\t{total_quantity}")
        current_product = product
        total_quantity = quantity

if current_product:
    print(f"{current_product}\t{total_quantity}")


Writing reducer2.py


In [15]:
!cat data.txt | grep -v "Cantidad" |python mapper2.py | sort -k1,1 |python reducer2.py

ProductoA	10
ProductoB	9
ProductoC	6
ProductoD	1


## Puntajes de Peliculas.

In [16]:
%%writefile movie_data.txt
UserID,MovieID,Rating,Timestamp
1,101,4,964982703
1,102,3,964982703
1,103,5,964982703
2,101,5,964982703
2,103,2,964982703
2,104,4,964982703
3,102,4,964982703
3,104,3,964982703
3,105,2,964982703
4,101,3,964982703
4,105,5,964982703
4,106,4,964982703
5,102,4,964982703
5,106,5,964982703
5,107,2,964982703
6,101,4,964982703
6,103,3,964982703
6,108,5,964982703
7,102,5,964982703
7,108,4,964982703
7,109,3,964982703
8,101,2,964982703
8,103,4,964982703
8,110,5,964982703
9,102,4,964982703
9,110,3,964982703
9,111,2,964982703
10,101,5,964982703
10,111,3,964982703
10,112,4,964982703
11,103,5,964982703
11,112,4,964982703
11,113,3,964982703
12,101,2,964982703
12,113,5,964982703
12,114,4,964982703
13,102,3,964982703
13,114,5,964982703
13,115,4,964982703
14,101,4,964982703
14,103,5,964982703
14,115,3,964982703
15,102,2,964982703
15,116,4,964982703
15,117,5,964982703
16,101,3,964982703
16,116,4,964982703
16,118,5,964982703
17,102,4,964982703
17,118,3,964982703
17,119,2,964982703
18,101,5,964982703
18,103,4,964982703
18,119,3,964982703
19,102,3,964982703
19,119,4,964982703
19,120,5,964982703
20,101,2,964982703
20,103,5,964982703
20,120,4,964982703
21,102,4,964982703
21,120,5,964982703
21,121,3,964982703
22,101,3,964982703
22,121,5,964982703
22,122,4,964982703
23,102,5,964982703
23,122,3,964982703
23,123,2,964982703
24,101,4,964982703
24,123,5,964982703
24,124,3,964982703
25,102,2,964982703
25,124,4,964982703
25,125,5,964982703

Writing movie_data.txt


In [19]:
%%writefile mapper3.py

#!/usr/bin/env python
import sys

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    # Divide la línea en campos
    fields = line.strip().split(",")
    if len(fields) == 4:
        user, movie, rating, timestamp = fields
        # Emite el MovieID como clave y la calificación como valor
        print(f"{movie}\t{rating}")


Writing mapper3.py


In [17]:
%%writefile reducer3.py
#!/usr/bin/env python
import sys

current_movie = None
total_rating = 0
total_reviews = 0

# Lee líneas de entrada desde la entrada estándar
for line in sys.stdin:
    movie, rating = line.strip().split("\t")
    rating = int(rating)

    if movie == current_movie:
        total_rating += rating
        total_reviews += 1
    else:
        if current_movie:
            average_rating = total_rating / total_reviews
            print(f"{current_movie}\t{average_rating:.2f}")
        current_movie = movie
        total_rating = rating
        total_reviews = 1

if current_movie:
    average_rating = total_rating / total_reviews
    print(f"{current_movie}\t{average_rating:.2f}")


Writing reducer3.py


In [21]:
!cat movie_data.txt  | python mapper3.py |grep -v "Movie" | sort -k1,1 | python reducer3.py

101	3.54
102	3.58
103	4.12
104	3.50
105	3.50
106	4.50
107	2.00
108	4.50
109	3.00
110	4.00
111	2.50
112	4.00
113	4.00
114	4.50
115	3.50
116	4.00
117	5.00
118	4.00
119	3.00
120	4.67
121	4.00
122	3.50
123	3.50
124	3.50
125	5.00


## Multiplicacion de matrices con Hadoop

In [22]:
!seq 1 100 | awk '{i++;print "A,"i","int(rand()*20); if(i==10){i=0}}' > matrix_in.txt
!seq 1 100 | awk '{i++;print "B,"i","int(rand()*20); if(i==10){i=0}}' >> matrix_in.txt
! head matrix_in.txt
! tail matrix_in.txt

A,1,6
A,2,5
A,3,19
A,4,2
A,5,5
A,6,0
A,7,4
A,8,13
A,9,8
A,10,16
B,1,2
B,2,10
B,3,5
B,4,19
B,5,5
B,6,0
B,7,14
B,8,19
B,9,19
B,10,16


# Nextflow

In [23]:
!java -version

[0.038s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.038s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [24]:
!curl -s https://get.nextflow.io | bash

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

      N E X T F L O W
      version 26.10.6 build 15032
      created 12-11-2026 23:21 UTC 
      cite doi:10.1038/nbt.3820
      http://nextflow.io


Nextflow installation completed. Please note:
- the executable file `nextflow` has been created in the folder: /content
- you may complete the installation by moving it to a directory in your $PATH



In [25]:
! /content/nextflow run hello

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

 N E X T F L O W   ~  version 26.10.6

Pulling nextflow-io/hello ...
 downloaded from https://github.com/nextflow-io/hello.git
Launching `https://github.com/nextflow-io/hello` [sleepy_allen] revision: 3c2cdc9823 [master]

[-        ] sayHello -

[-        ] sayHello | 0 of 4

executor >  local (2)
[]8;;file:///content/work/8a/ab131baf8377818d0ca7923e5395188a/ab131b]8;;] sayHello (2) | 0 of 4

executor >  local (4)
[]8;;file:///content/work/40/c7eb3c68bdf514d3aeb63be055434b40/c7eb3c]8;;] sayHello (4) | 0 of 4
Bonjour world!

Ciao world!

Hello world!


executor >  local (4)
[]8;;file:///content/work/40/c7eb3c68bdf514d3aeb63be055434b40/c7eb3c]8;;] sayH

In [31]:
%%writefile sayhello.nf

nextflow.enable.dsl=2
process sayHello {
    input:
        val cheers

    output:
        stdout emit: verbiage

    script:
    """
    echo -n $cheers
    """
}

workflow {
    things = channel.of('Hola Mundo!', 'Hello world!', 'Bazinga!!', 'Hello PMD')
    things.view()
    sayHello(things)
    sayHello.out.verbiage.view()
}

Overwriting sayhello.nf


In [32]:
! /content/nextflow run  sayhello.nf

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

 N E X T F L O W   ~  version 26.10.6

Launching `sayhello.nf` [cranky_neumann] revision: ac11e06269

[-        ] sayHello -
Hola Mundo!
Hello world!
Bazinga!!
Hello PMD

[-        ] sayHello | 0 of 4
Hola Mundo!
Hello world!
Bazinga!!
Hello PMD

executor >  local (2)
[]8;;file:///content/work/48/95490eeb356300af7ebb157be02abe48/95490e]8;;] sayHello (2) | 0 of 4
Hola Mundo!
Hello world!
Bazinga!!
Hello PMD

executor >  local (4)
[]8;;file:///content/work/a0/0187a9965d46b883222623ddabc08ea0/0187a9]8;;] sayHello (4) | 4 of 4 ✔
Hola Mundo!
Hello world!
Bazinga!!
Hello PMD
Hello world!
Bazinga!!
Hola Mundo!
Hello PMD

executor >  local (4)
[]8;;file:///cont

## Singularity configuration

In [33]:
!add-apt-repository -y ppa:apptainer/ppa

Repository: 'Types: deb
URIs: https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu/
Suites: noble
Components: main
'
Description:
This PPA contains stable releases of Apptainer.

Documentation https://apptainer.org/docs/
Support https://apptainer.org/help/
Bug report https://github.com/apptainer/apptainer/issues
More info: https://launchpad.net/~apptainer/+archive/ubuntu/ppa
Adding repository.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.7 kB]
Get:9 http://archive.ubun

In [34]:
!apt update

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu noble InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,822 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,184 kB]
Fetched 4,133 kB in 32s (129 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
64 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of co

In [35]:
! apt install -y apptainer

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fakeroot libfakeroot libfuse3-3 libprotobuf-c1 libsubid4 libtalloc2 uidmap
Suggested packages:
  fuse3
The following NEW packages will be installed:
  apptainer fakeroot libfakeroot libfuse3-3 libprotobuf-c1 libsubid4
  libtalloc2 uidmap
0 upgraded, 8 newly installed, 0 to remove and 64 not upgraded.
Need to get 37.7 MB of archives.
After this operation, 160 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu noble/main amd64 apptainer amd64 1.5.5-1~noble [37.4 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libfuse3-3 amd64 3.14.0-5build1 [83.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 libprotobuf-c1 amd64 1.4.1-1ubuntu4 [20.8 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 libtalloc2 amd64 2.4.2-1build2 [27.3 kB]
Get:5 http://archive.ubuntu.com

In [66]:
%%writefile channel_play.nf

nextflow.enable.dsl=2
/*
process sayHello {
    input:
        val cheers

    output:
        stdout emit: verbiage

    script:
    """
    echo -n $cheers
    """
}
*/
//proceso cuenta las linas y caracteres de los archivos .txt y .bin
//generando un archivo llamado id.nl
process COUNT_LINE{
  publishDir "./results/count_line", mode: 'copy'



  input:
   tuple val(id),path(txt), path(bin)

  output:
   tuple val(id), file("${id}.nl")

   script:
   """
   wc -l ${txt} > ${id}.nl
   wc -c ${bin} >> ${id}.nl
   """
}


workflow {
    c_txt = channel.fromPath("/content/files/*.txt")
    .map{ row -> tuple(row.baseName, file(row))}
    //.view()
    //c_txt.view()
    c_bin= channel.fromPath("/content/files/*.bin")
    .map{ row -> tuple(row.baseName, file(row))}


    //c_txt.view()
    //c_bin.view()

    merge_t=c_txt.join(c_bin)
    //merge by key
    //merge_t.view()
    //llamamos al proceso con el canal que construimos.
    COUNT_LINE(merge_t)
    //c_bin.view()
    //things.view()
   // sayHello(things)
   // sayHello.out.verbiage.view()
}

Overwriting channel_play.nf


In [68]:
! /content/nextflow run channel_play.nf -resume

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

 N E X T F L O W   ~  version 26.10.6

Launching `channel_play.nf` [distracted_tesla] revision: ec3fbf0203

[-        ] COUNT_LINE -

[]8;;file:///content/work/53/b4cfd9f04bc9de53a23abedfa18dd453/b4cfd9]8;;] COUNT_LINE (2) | 2 of 2, cached: 2

[]8;;file:///content/work/98/df09f787709dd6d21a381641bea2fb98/df09f7]8;;] COUNT_LINE (4) | 4 of 4, cached: 4

[]8;;file:///content/work/5a/65743a8a58faf6b945aff1e91a68e35a/65743a]8;;] COUNT_LINE (10) | 10 of 10, cached: 10

[]8;;file:///content/work/9e/f3408fcadd0ae4847fa31ba78000cf9e/f3408f]8;;] COUNT_LINE (15) | 16 of 16, cached: 16

[]8;;file:///content/work/c8/f2c3fa8b8a79eda32e1cb5a96365ecc8/f2c3fa]

In [ ]:
! apt install -y apptainer-suid

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  apptainer-suid
0 upgraded, 1 newly installed, 0 to remove and 22 not upgraded.
Need to get 5,600 kB of archives.
After this operation, 17.5 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu jammy/main amd64 apptainer-suid amd64 1.2.4-1~ubuntu22.04.0+stable1 [5,600 kB]
Fetched 5,600 kB in 2s (2,848 kB/s)
Selecting previously unselected package apptainer-suid.
(Reading database ... 122584 files and directories currently installed.)
Preparing to unpack .../apptainer-suid_1.2.4-1~ubuntu22.04.0+stable1_amd64.deb ...
Unpacking apptainer-suid (1.2.4-1~ubuntu22.04.0+stable1) ...
Setting up apptainer-suid (1.2.4-1~ubuntu22.04.0+stable1) ...


In [ ]:
! singularity

Usage:
  apptainer [global options...] <command>

Available Commands:
  build       Build an Apptainer image
  cache       Manage the local cache
  capability  Manage Linux capabilities for users and groups
  checkpoint  Manage container checkpoint state (experimental)
  completion  Generate the autocompletion script for the specified shell
  config      Manage various apptainer configuration (root user only)
  delete      Deletes requested image from the library
  exec        Run a command within a container
  inspect     Show metadata for an image
  instance    Manage containers running as services
  key         Manage OpenPGP keys
  oci         Manage OCI containers
  overlay     Manage an EXT3 writable overlay image
  plugin      Manage Apptainer plugins
  pull        Pull an image from a URI
  push        Upload image to the provided URI
  remote      Manage apptainer remote endpoints, keyservers and OCI/Docker registry credentials
  run         Run the user-defined default comman

# Examples

In [ ]:
! wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR134/021/SRR13447321/SRR13447321_1.fastq.gz
! wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR134/021/SRR13447321/SRR13447321_2.fastq.gz

--2023-10-25 14:33:35--  ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR134/021/SRR13447321/SRR13447321_1.fastq.gz
           => ‘SRR13447321_1.fastq.gz’
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /vol1/fastq/SRR134/021/SRR13447321 ... done.
==> SIZE SRR13447321_1.fastq.gz ... 693436099
==> PASV ... done.    ==> RETR SRR13447321_1.fastq.gz ... done.
Length: 693436099 (661M) (unauthoritative)

SRR13447321_1.fastq 100%[===================>] 661.31M  24.6MB/s    in 26s     

2023-10-25 14:34:04 (25.6 MB/s) - ‘SRR13447321_1.fastq.gz’ saved [693436099]

--2023-10-25 14:34:04--  ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR134/021/SRR13447321/SRR13447321_2.fastq.gz
           => ‘SRR13447321_2.fastq.gz’
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.

In [ ]:
%%writefile ejemplo1.nf

// Codigo nextflow
nextflow.enable.dsl=2
params.files=""
process NUM_LINEAS {
    input:
    path read

    script:
    """
    printf '${read} '
    gunzip -c ${read} | wc -l
    """
}

reads_ch = Channel.fromPath(params.files)

workflow {
  NUM_LINEAS(reads_ch)
}

Overwriting ejemplo1.nf


In [ ]:
! /content/nextflow run  ejemplo1.nf --files "*.fastq.gz"  -process.echo

N E X T F L O W  ~  version 23.10.0
Launching `ejemplo1.nf` [magical_keller] DSL2 - revision: ab4c064c38
[-        ] process > NUM_LINEAS -

executor >  local (1)
[-        ] process > NUM_LINEAS [  0%] 0 of 2

executor >  local (2)
[10/833b31] process > NUM_LINEAS (2) [  0%] 0 of 2

executor >  local (2)
[10/833b31] process > NUM_LINEAS (2) [ 50%] 1 of 2
SRR13447321_1.fastq.gz 37769292


executor >  local (2)
[33/21ed7e] process > NUM_LINEAS (1) [100%] 2 of 2 ✔
SRR13447321_1.fastq.gz 37769292

SRR13447321_2.fastq.gz 37769292




In [ ]:
%%writefile fastqc.nf

process FASTQC {
    tag "$sampleId-mem"
    //label 'process_medium'
    publishDir "$params.outdir/QC/FASTQC", mode: "copy"

  container "${ workflow.containerEngine == 'singularity' && !task.ext.singularity_pull_docker_container ?
        'https://depot.galaxyproject.org/singularity/fastqc:0.12.1--hdfd78af_0' :
        'biocontainers/fastqc:0.12.1--hdfd78af_0' }"

    input:
    tuple val(sampleId), val(part), file(read1), file(read2)

    output:
    path("${sampleId}-${part}.fastqc"), emit: fqc

    script:
    if(params.debug == true){
    """
    echo fastqc -o ${sampleId}-${part}.fastqc $read1 $read2
    mkdir -p ${sampleId}-${part}.fastqc
    touch ${sampleId}-${part}.fastqc/report.fastqc

    """
    } else{
    """
    mkdir -p ${sampleId}-${part}.fastqc
    fastqc -t $task.cpus -o ${sampleId}-${part}.fastqc $read1 $read2
    """
    }

}
workflow {
    if(params.reads != null){
    // --reads "./reads/B087*.merge.{1,2}.fq.gz"
    read_pairs_ch = channel.fromFilePairs(params.reads)
    }else if(params.csv != null){
    //we reads pairs from csv
    read_pairs_ch=Channel.fromPath(params.csv) \
        | splitCsv(header:true) \
        | map { row-> tuple(row.sampleId, row.part,  file(row.read1), file(row.read2)) }
    }else{
        println "Error: reads regex or path"
    }

    read_pairs_ch.view()
    FASTQC(read_pairs_ch)

}


Writing fastqc.nf


In [ ]:
%%writefile input.csv
sampleId,part,read1,read2
test,0,/content/SRR13447321_1.fastq.gz,/content/SRR13447321_2.fastq.gz

Overwriting input.csv


In [ ]:
! /content/nextflow run  fastqc.nf --csv "/content/input.csv"  -process.echo --debug true

N E X T F L O W  ~  version 23.10.0
Launching `fastqc.nf` [fabulous_swirles] DSL2 - revision: 2c67865c3e
WARN: Access to undefined parameter `reads` -- Initialise it to a default value eg. `params.reads = some_value`
[-        ] process > FASTQC -
[test, 0, /content/SRR13447321_1.fastq.gz, /content/SRR13447321_2.fastq.gz]

executor >  local (1)
[a1/ba98de] process > FASTQC (test-mem) [  0%] 0 of 1
[test, 0, /content/SRR13447321_1.fastq.gz, /content/SRR13447321_2.fastq.gz]

executor >  local (1)
[a1/ba98de] process > FASTQC (test-mem) [100%] 1 of 1 ✔
[test, 0, /content/SRR13447321_1.fastq.gz, /content/SRR13447321_2.fastq.gz]
fastqc -o test-0.fastqc SRR13447321_1.fastq.gz SRR13447321_2.fastq.gz

WARN: Access to undefined parameter `outdir` -- Initialise it to a default value eg. `params.outdir = some_value`
WARN: Process `FASTQC` publishDir path contains a variable with a null value

executor >  local (1)
[a1/ba98de] process > FASTQC (test-mem) [100%] 1 of 1 ✔
[test, 0, /content/SRR13447